# Analisis de datos VideoGame sales de Kaggle

In [1]:
# Librerias a usar

import kagglehub
import os
import pandas as pd

import plotly.express as px

Carga de archivos desde Kaggle

In [2]:
# 1. Descargar la última versión del dataset
path = kagglehub.dataset_download("volodymyrpivoshenko/video-game-sales-dataset")

print("Path to dataset files:", path)
print("Archivos descargados:", os.listdir(path))

# 2. Cargar el CSV principal (según Kaggle suele llamarse así)
csv_path = os.path.join(path, "video_games_sales.csv")
df = pd.read_csv(csv_path)

# 3. Revisar estructura básica
print(df.shape)
print(df.columns)
df.head()

Path to dataset files: /root/.cache/kagglehub/datasets/volodymyrpivoshenko/video-game-sales-dataset/versions/1
Archivos descargados: ['video_games_sales.csv']
(16598, 11)
Index(['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales'],
      dtype='object')


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [3]:
# pasamos todos los numbres de las columnas a minusculas

df.columns = df.columns.str.lower()
df.columns


Index(['rank', 'name', 'platform', 'year', 'genre', 'publisher', 'na_sales',
       'eu_sales', 'jp_sales', 'other_sales', 'global_sales'],
      dtype='object')

In [4]:
# revisamos los tipos de datos

df.dtypes

,0
rank,int64
name,object
platform,object
year,float64
genre,object
publisher,object
na_sales,float64
eu_sales,float64
jp_sales,float64
other_sales,float64


In [5]:
# revisamos valore unicos en year

df["year"].unique()

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2003., 1986., 2000.,   nan, 1995., 2016., 1991., 1981., 1987.,
       1980., 1983., 2020., 2017.])

In [6]:
#Luego la convertimos a entero nullable (Int64), que soporta NaN:

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

df["year"].dtype
df["year"].head(10)


,year
0,2006
1,1985
2,2008
3,2009
4,1996
5,1989
6,2006
7,2006
8,2009
9,1984


In [7]:
# Valores únicos en variables categóricas

df["platform"].nunique(), df["platform"].unique()


(31,
 array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
        '3DS', 'PS4', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne', 'GC',
        'WiiU', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
        '3DO', 'GG', 'PCFX'], dtype=object))

In [8]:
sorted(df["platform"].unique())


['2600',
 '3DO',
 '3DS',
 'DC',
 'DS',
 'GB',
 'GBA',
 'GC',
 'GEN',
 'GG',
 'N64',
 'NES',
 'NG',
 'PC',
 'PCFX',
 'PS',
 'PS2',
 'PS3',
 'PS4',
 'PSP',
 'PSV',
 'SAT',
 'SCD',
 'SNES',
 'TG16',
 'WS',
 'Wii',
 'WiiU',
 'X360',
 'XB',
 'XOne']

In [9]:
# Publisher

df["publisher"].nunique()
df["publisher"].value_counts().head(20)


,count
publisher,
Electronic Arts,1351
Activision,975
Namco Bandai Games,932
Ubisoft,921
Konami Digital Entertainment,832
THQ,715
Nintendo,703
Sony Computer Entertainment,683
Sega,639


In [10]:
# Chequeo de consistencias de Ventas

df["ventas_regiones"] = (
    df["na_sales"] + df["eu_sales"] + df["jp_sales"] + df["other_sales"]
)

(df["global_sales"] - df["ventas_regiones"]).describe()


,0
count,16598.000000
mean,0.000277
std,0.005223
min,-0.020000
25%,0.000000
50%,0.000000
75%,0.000000
max,0.020000


La diferencia entre global_sales y la suma de las 4 regiones es prácticamente cero.
El mínimo es -0.02 y el máximo 0.02 → o sea, descalce de ±0.02 millones de copias (±20.000 unidades).
Mediana 0, 75% de los casos 0 → en la mayoría de los juegos la suma de regiones calza perfecto con global_sales.
Eso es puro redondeo del dataset, nada raro.
Conclusión:
✅ global_sales es consistente con la suma de NA_Sales + EU_Sales + JP_Sales + Other_Sales.
Puedes confiar en esa métrica sin drama.

In [11]:
#Revisamos nulos por columna

df.isna().sum()


,0
rank,0
name,0
platform,0
year,271
genre,0
publisher,58
na_sales,0
eu_sales,0
jp_sales,0
other_sales,0


In [12]:
# Distribucion numerica

df.describe()


,rank,year,na_sales,eu_sales,jp_sales,other_sales,global_sales,ventas_regiones
count,16598.000000,16327.0,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441,0.537164
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028,1.555151
min,1.000000,1980.0,0.000000,0.000000,0.000000,0.000000,0.010000,0.000000
25%,4151.250000,2003.0,0.000000,0.000000,0.000000,0.000000,0.060000,0.060000
50%,8300.500000,2007.0,0.080000,0.020000,0.000000,0.010000,0.170000,0.170000
75%,12449.750000,2010.0,0.240000,0.110000,0.040000,0.040000,0.470000,0.470000
max,16600.000000,2020.0,41.490000,29.020000,10.220000,10.570000,82.740000,82.740000


In [13]:
# Chequeamos años faltantes que sen raros

df["year"].value_counts(dropna=False).sort_index()


,count
year,
1980,9
1981,46
1982,36
1983,17
1984,14
1985,14
1986,21
1987,16
1988,15


Ese value_counts() dice esto en simple:
Datos desde 1980 hasta 2016 con buena densidad.
Muy pocos registros en 2017 (3 juegos) y 2020 (1 juego).
271 filas sin año (<NA>).
Total de filas: ~16.598 → los <NA> son como un 1.6% del dataset, nada dramático.
1️⃣ Qué nos dice esto para el EDA
El “corazón” del dataset es 1980–2016.
Los años 2017 y 2020 son claramente residuales (pueden ser ports, errores o simplemente muy pocos datos).
Los <NA> en year no sirven para análisis temporal, pero sí podrían usarse en análisis que no involucren año (ej. top géneros, top publishers).
2️⃣ Estrategia limpia (sin enredarse)
Lo más ordenado es:
Mantener df como dataframe original.
Crear un dataframe aparte solo para análisis por año, filtrando:
sin nulos en year,
años entre 1980 y 2016 (el bloque fuerte de datos).

In [14]:
# nos quedamos solo con los años 1980 a 2016 para un analsisi temporal serio

# 1. Filtrar filas con año conocido
df_year = df[df["year"].notna()].copy()

# 2. Quedarse solo con el rango “sano” 1980–2016
df_year = df_year[df_year["year"].between(1980, 2016)]

df_year["year"].min(), df_year["year"].max(), df_year.shape

(np.int64(1980), np.int64(2016), (16323, 12))

# Ventas globales por año

In [15]:
ventas_por_anio = (
    df_year.groupby("year")["global_sales"]
           .sum()
           .reset_index()
           .sort_values("year")
)
ventas_por_anio.head()


,year,global_sales
0,1980,11.38
1,1981,35.77
2,1982,28.86
3,1983,16.79
4,1984,50.36


In [24]:
# Línea interactiva

fig_year = px.line(
    ventas_por_anio,
    x="year",
    y="global_sales_m",
    markers=True,
    title="Evolución de las ventas globales de videojuegos (1980–2016)",
    labels={
        "year": "Año",
        "global_sales_m": "Ventas globales (millones de copias)"
    }
)

fig_year.update_layout(
    template="plotly_white",
    title_font=dict(size=24),
    font=dict(family="Arial", size=14),
    hovermode="x unified",
    margin=dict(l=60, r=40, t=80, b=60),
    width=1000,
    height=450
)

# Quitar el range slider (el mini gráfico de abajo)
fig_year.update_xaxes(
    rangeslider_visible=False,
    dtick=2
)

# Darle más presencia a la línea
fig_year.update_traces(
    line=dict(width=3),
    marker=dict(size=6),
    hovertemplate="<b>Año:</b> %{x}<br>" +
                  "<b>Ventas globales:</b> %{y:.1f} M copias<extra></extra>"
)

# Anotar el pico (2008–2009)
peak_row = ventas_por_anio.loc[ventas_por_anio["global_sales_m"].idxmax()]
fig_year.add_annotation(
    x=int(peak_row["year"]),
    y=peak_row["global_sales_m"],
    text="Pico de ventas<br>(era PS3 / Xbox 360 / Wii)",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40
)

fig_year.show()


# Ventas globales por plataforma

In [16]:
ventas_por_plataforma = (
    df.groupby("platform")["global_sales"]
      .sum()
      .reset_index()
      .sort_values("global_sales", ascending=False)
)
ventas_por_plataforma.head(10)


,platform,global_sales
16,PS2,1255.64
28,X360,979.96
17,PS3,957.84
26,Wii,926.71
4,DS,822.49
15,PS,730.66
6,GBA,318.50
19,PSP,296.28
18,PS4,278.10
13,PC,258.82


In [21]:
# Agregar ventas globales por plataforma
ventas_por_plataforma = (
    df.groupby("platform", as_index=False)["global_sales"]
      .sum()
      .rename(columns={"global_sales": "global_sales_m"})
)

# Top 10 plataformas
top_platforms = (
    ventas_por_plataforma
    .sort_values("global_sales_m", ascending=False)
    .head(10)
)

# Barra horizontal
fig_platform = px.bar(
    top_platforms.sort_values("global_sales_m"),  # ordenar para barra horizontal
    x="global_sales_m",
    y="platform",
    orientation="h",
    title="Top 10 plataformas por ventas globales acumuladas",
    labels={
        "platform": "Plataforma",
        "global_sales_m": "Ventas globales (millones de copias)"
    }
)

fig_platform.update_layout(
    template="plotly_white",
    title_font=dict(size=22),
    font=dict(family="Arial", size=14),
    margin=dict(l=90, r=40, t=80, b=60)
)

fig_platform.update_traces(
    hovertemplate="<b>Plataforma:</b> %{y}<br>" +
                  "<b>Ventas globales:</b> %{x:.1f} M copias<extra></extra>"
)

fig_platform.show()


# Ventas globales por genero

In [17]:
ventas_por_genero = (
    df.groupby("genre")["global_sales"]
      .sum()
      .reset_index()
      .sort_values("global_sales", ascending=False)
)
ventas_por_genero


,genre,global_sales
0,Action,1751.18
10,Sports,1330.93
8,Shooter,1037.37
7,Role-Playing,927.37
4,Platform,831.37
3,Misc,809.96
6,Racing,732.04
2,Fighting,448.91
9,Simulation,392.20
5,Puzzle,244.95


In [23]:
ventas_por_genero = (
    df.groupby("genre", as_index=False)["global_sales"]
      .sum()
      .rename(columns={"global_sales": "global_sales_m"})
)

# Ordenar por ventas
ventas_por_genero = ventas_por_genero.sort_values("global_sales_m", ascending=False)

fig_genre = px.bar(
    ventas_por_genero,
    x="genre",
    y="global_sales_m",
    title="Ventas globales por género de videojuego",
    labels={
        "genre": "Género",
        "global_sales_m": "Ventas globales (millones de copias)"
    }
)

fig_genre.update_layout(
    template="plotly_white",
    title_font=dict(size=22),
    font=dict(family="Arial", size=14),
    xaxis_tickangle=-45,
    margin=dict(l=60, r=40, t=80, b=100)
)

fig_genre.update_traces(
    hovertemplate="<b>Género:</b> %{x}<br>" +
                  "<b>Ventas globales:</b> %{y:.1f} M copias<extra></extra>"
)

fig_genre.show()


## Analizamos el año 2008 para entender las ventas tan altas

In [25]:
# filtramos 2008

df_2008 = df_year[df_year["year"] == 2008].copy()

df_2008.shape        # nº de filas (juegos)
df_2008["global_sales"].sum()


np.float64(678.8999999999999)

In [26]:
# vemos las plataformas protagonistas

platform_2008 = (
    df_2008.groupby("platform", as_index=False)["global_sales"]
           .sum()
           .sort_values("global_sales", ascending=False)
)
platform_2008


,platform,global_sales
6,Wii,174.16
1,DS,147.89
7,X360,135.76
4,PS3,119.69
3,PS2,53.83
5,PSP,34.68
2,PC,12.67
8,XB,0.18
0,DC,0.04


Wii → ~174.2 M
DS → ~147.9 M
Xbox 360 → ~135.8 M
PS3 → ~119.7 M
Luego PS2, PSP, PC muy por detrás
Traducción: en 2008 tienes la tormenta perfecta de la generación Wii/DS + Xbox 360 + PS3 ya maduras.
Wii y DS juntas concentran casi la mitad de las ventas del año.

## ¿Qué tipo de juegos? – Géneros

In [27]:
genre_2008 = (
    df_2008.groupby("genre", as_index=False)["global_sales"]
           .sum()
           .sort_values("global_sales", ascending=False)
)
genre_2008


,genre,global_sales
0,Action,136.39
10,Sports,95.34
3,Misc,87.03
6,Racing,70.66
7,Role-Playing,59.83
8,Shooter,59.51
9,Simulation,46.76
4,Platform,35.70
2,Fighting,35.38
1,Adventure,25.02


## ¿Quiénes se llevaron el cheque? – Publishers

In [28]:
publisher_2008 = (
    df_2008.groupby("publisher", as_index=False)["global_sales"]
           .sum()
           .sort_values("global_sales", ascending=False)
           .head(10)
)
publisher_2008


,publisher,global_sales
108,Nintendo,91.22
52,Electronic Arts,84.12
6,Activision,67.41
151,Ubisoft,57.44
142,Take-Two Interactive,46.18
131,Sega,37.19
140,THQ,30.45
83,Konami Digital Entertainment,27.82
132,Sony Computer Entertainment,26.64
48,Disney Interactive Studios,22.09


## Top juegos 2008

In [29]:
top_games_2008 = (
    df_2008.sort_values("global_sales", ascending=False)
           [["name", "platform", "genre", "publisher", "global_sales"]]
           .head(15)
)
top_games_2008


,name,platform,genre,publisher,global_sales
2,Mario Kart Wii,Wii,Racing,Nintendo,35.82
39,Super Smash Bros. Brawl,Wii,Fighting,Nintendo,13.04
51,Grand Theft Auto IV,X360,Action,Take-Two Interactive,11.02
56,Grand Theft Auto IV,PS3,Action,Take-Two Interactive,10.57
88,Pokémon Platinum Version,DS,Role-Playing,Nintendo,7.84
98,Call of Duty: World at War,X360,Shooter,Activision,7.37
118,Gears of War 2,X360,Shooter,Microsoft Game Studios,6.76
144,Metal Gear Solid 4: Guns of the Patriots,PS3,Action,Konami Digital Entertainment,6.03
148,LittleBigPlanet,PS3,Platform,Sony Computer Entertainment,5.92
161,Monster Hunter Freedom Unite,PSP,Role-Playing,Capcom,5.50


2008 es el punto máximo porque coincide la madurez de las consolas de séptima generación (Wii, DS, Xbox 360, PS3) con una ola de blockbusters y juegos familiares que amplían masivamente la base de jugadores.

In [30]:
# 1) Filtramos 2008
df_2008 = df_year[df_year["year"] == 2008].copy()

# 2) Nos quedamos con los publishers más grandes para no saturar
top_pubs_2008 = (
    df_2008.groupby("publisher")["global_sales"]
           .sum()
           .sort_values(ascending=False)
           .head(8)           # top 8 publishers
           .index
)

df_2008_top = df_2008[df_2008["publisher"].isin(top_pubs_2008)].copy()

# 3) Treemap: plataforma -> género -> publisher
fig_2008_treemap = px.treemap(
    df_2008_top,
    path=["platform", "genre", "publisher"],
    values="global_sales",
    title="Ventas de videojuegos en 2008 por plataforma, género y publisher (Top publishers)",
    labels={
        "platform": "Plataforma",
        "genre": "Género",
        "publisher": "Publisher",
        "global_sales": "Ventas globales (M de copias)"
    }
)

fig_2008_treemap.update_layout(
    template="plotly_white",
    title_font=dict(size=22),
    font=dict(family="Arial", size=14),
    margin=dict(l=10, r=10, t=60, b=10),
    width=900,
    height=500
)

fig_2008_treemap.update_traces(
    hovertemplate="<b>Plataforma:</b> %{label}<br>" +
                  "<b>Ruta:</b> %{currentPath}<br>" +
                  "<b>Ventas:</b> %{value:.1f} M copias<extra></extra>"
)

fig_2008_treemap.show()

Wii y DS ocupan medio mapa.
Acción / deportes / “misc” dominan.
Nintendo, EA, Activision y compañía se reparten los bloques grandes.